In [1]:
# 0. Environment Setup

# Clone the gemma repository
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository for UI/UX
!git clone https://github.com/google-deepmind/dialog.git || true
!pip install -q ./dialog

# Ensure local modules are in path
import sys
import os
sys.path.append(os.getcwd())

Cloning into 'gemma'...
remote: Enumerating objects: 2502, done.
remote: Counting objects: 100% (1117/1117), done.
remote: Compressing objects: 100% (376/376), done.
remote: Total 2502 (delta 898), reused 745 (delta 741), pack-reused 1385 (from 3)
Receiving objects: 100% (2502/2502), 1.20 MiB | 4.88 MiB/s, done.
Resolving deltas: 100% (1753/1753), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 73.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.4/318.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.8/633.8 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.6/619.6 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 25.3 MB/s 

In [2]:
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

Cloning into 'Titans_jax'...
remote: Enumerating objects: 466, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 466 (delta 10), reused 13 (delta 9), pack-reused 449 (from 1)
Receiving objects: 100% (466/466), 157.56 MiB | 24.50 MiB/s, done.
Resolving deltas: 100% (271/271), done.
/content/Titans_jax


In [ ]:
def load_params_for_sampling(workdir: str, state_template: kd.train.TrainState):
    """
    Извлекает только веса (params) из последнего полного чекпоинта обучения
    для использования в сэмплере.
    """
    import orbax.checkpoint as ocp
    checkpoint_path = os.path.join(workdir, 'checkpoints')
    
    mngr = ocp.CheckpointManager(os.path.abspath(checkpoint_path), ocp.StandardCheckpointer())
    latest_step = mngr.latest_step()
    
    if latest_step is None:
        print("⚠️ Чекпоинты не найдены! Сэмплер будет использовать случайные веса.")
        return state_template.params

    # Восстанавливаем только структуру state, но забираем из неё .params
    restored_state = mngr.restore(latest_step, items=state_template)
    print(f"🎯 Веса для сэмплера успешно загружены из чекпоинта (шаг {latest_step})")
    return restored_state.params

# --- ИСПОЛЬЗОВАНИЕ ---
# 1. Создаем шаблон состояния (без запуска обучения)
state_template = trainer.init_state()

# 2. Загружаем в него веса из папки обучения
sampling_params = load_params_for_sampling(trainer.workdir, state_template)

# 3. Теперь передаем эти params в твой цикл генерации
# (обычно это делается через model.apply(sampling_params, ... внутри сэмплера))


In [3]:
import importlib
import jax
import os
import gemma_titans
# importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans
import titans_tree_utils
from gemma import gm

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"

# jax.config.update("jax_disable_jit", True) # Temporarily disable JIT to bypass the hashing error

JAX Backend: gpu
Devices: [CudaDevice(id=0)]


In [17]:
del Gemma3_1B_Titans

In [18]:
importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config

In [19]:
import dataclasses
inference_config = Gemma_Titans_Config(
    **{f.name: getattr(Gemma3_1B_Titans.config, f.name) for f in dataclasses.fields(Gemma_Titans_Config) if f.name != 'is_training_mode'},
    is_training_mode=False # <--- ВЫКЛЮЧАЕМ ТРЕНИРОВКУ ДЛЯ СЭМПЛЕРА
)

In [20]:
import google.colab
import shutil
import orbax.checkpoint as ocp

if os.path.exists('/content/Titans_jax/saved_titans_delta.zip'):
    shutil.unpack_archive('/content/Titans_jax/saved_titans_delta.zip',"./saved_titans_delta")

def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()
    return checkpointer.restore(load_path)

import jax.numpy as jnp
# gemma_config = Gemma3_1B_Titans.config
# inference_config = Gemma_Titans_Config(
#     **{f.name: getattr(Gemma3_1B_Titans.config, f.name) for f in dataclasses.fields(_config.TransformerConfig)},
#     is_training_mode=False # <--- ВЫКЛЮЧАЕМ ТРЕНИРОВКУ ДЛЯ СЭМПЛЕРА
# )

# gemma_config.is_training_mode = False

model = Gemma3_1B_Titans(
    config=inference_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.tokens",
)

print("Loading weights...")
original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)
loaded_titans_params = load_titans_weights("./saved_titans_delta")

# Снимаем обертку Kauldron, если она есть
if 'model' in loaded_titans_params: # and 'blocks' not in loaded_titans_params:
    print("Removing Kauldron 'model' wrapper from loaded weights...")
    loaded_titans_params = loaded_titans_params['model']

merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)

# Жесткая проверка: убеждаемся, что веса реально вшились в рабочую ветку
assert 'memory_gate' in merged_params['layer_5'], "FATAL: Titans weights failed to merge!"
print("Weights merged successfully!")


tokenizer = gm.text.Gemma3Tokenizer()

Loading weights...


Weights merged successfully!


In [21]:
merged_params['layer_5'].keys()

dict_keys(['attn', 'mlp', 'post_attention_norm', 'post_ffw_norm', 'pre_attention_norm', 'pre_ffw_norm', 'memory', 'memory_gate'])

## 4. Interactive Dialogue with `google-deepmind/dialog`

In [22]:
import dialog
from gemma import gm
import jax

# Initialize Sampler and Conversation
sampler = gm.text.Sampler(
    model=model,
    params=merged_params,
    tokenizer=tokenizer,
)

conv = dialog.Conversation()


def chat(user_input: str):
    global conv
    # Add user message
    conv += dialog.User(user_input)

    # Convert conversation to prompt (Gemma 3 format)
    prompt = conv.as_text(training=False)

    # Generate response
    # Note: Sampler handles the caching of Titans memory automatically
    response_text = sampler.sample(prompt, max_new_tokens=128)

    # Add model response to UI
    conv += dialog.Model(response_text)
    conv.show()

# Example usage:
# chat("Привет! Кто такие Титаны в мифологии?")

In [ ]:
chat("Who is Titans in the ancient greek mythology?")
